# Harbor Pump Works: can we ship December's orders?

This notebook is for the buyer, the customer service lead and the plant manager. You do
not need to write code. Every gray box below is either a command you can run, or a file
you can read. Run the boxes from top to bottom. The answers are already saved under each box.

## (a) Your business

Harbor Pump Works builds two water pumps in Fremont, Nebraska:

- **PMP-100**, a 1/2 hp home well pump. It sells for $385.
- **PMP-300**, a 2 hp farm irrigation pump. It sells for $1,240.

Each pump is a motor, a cast housing, an impeller and seal kits. Harbor buys every part.
The motors and impellers come from overseas through a West Coast port. Ships there have
been 2 to 10 days late this fall, and when the port is slow, both suppliers are slow
together. The housings come by truck from a foundry in Iowa.

It is Wednesday morning, November 18. Seven customer orders are due between now and
December 18, worth about $69,000. **The questions: which orders will ship on time, which
are at risk, and what should the buyer do today?** The buyer has a plan B: pay about
$3,800 to air-freight both motor orders, and place new orders for small housings and
impellers.

## (b) Your inputs

Every file in `data/` is an export a planner already has. Nothing here is made up by the tool.

| File | What it is | Where it comes from |
|---|---|---|
| `data/items.csv` | The item master: pumps and parts | ERP item master |
| `data/suppliers.csv` | Suppliers and how late they can be | Buyer's supplier scorecard |
| `data/on-hand.csv` | Stock on the shelf this morning | ERP inventory report |
| `data/open-purchase-orders.csv` | Parts on order, with promised dock dates | ERP open PO report |
| `data/customer-orders.csv` | Customer orders and due dates | ERP open sales orders |
| `harbor-pumps.twin.yaml` | The twin capsule: ties the files together and holds the bill of materials | You write it once |

### Items

In [1]:
!cat data/items.csv

id,kind,uom,description
PMP-100,assembly,each,Home well pump 1/2 hp
PMP-300,assembly,each,Farm irrigation pump 2 hp
MTR-05,purchased,each,Electric motor 1/2 hp
MTR-20,purchased,each,Electric motor 2 hp
HSG-100,purchased,each,Cast iron pump housing small
HSG-300,purchased,each,Cast iron pump housing large
IMP-100,purchased,each,Bronze impeller small
IMP-300,purchased,each,Bronze impeller large
SEAL-KIT,purchased,each,Shaft seal and gasket kit


| Column | Meaning |
|---|---|
| `id` | The part number. |
| `kind` | `assembly` is a pump we build. `purchased` is a part we buy. |
| `uom` | The unit: each. |
| `description` | A name a person can read. twinflow does not need it. |

### Suppliers

In [2]:
!cat data/suppliers.csv

id,site_id,delay_risk.minimum_days,delay_risk.maximum_days,delay_risk.common_risk_group
volt-motors,plant,2,10,west-coast-port
pacific-impeller,plant,1,8,west-coast-port
midwest-foundry,plant,0,3,midwest-trucking
lakeside-seals,plant,0,1,local-delivery
assembly-line,plant,,,
volt-motors-air,plant,0,2,air-freight


| Column | Meaning |
|---|---|
| `id` | The supplier. `volt-motors-air` is Volt Motors shipping by air (plan B only). |
| `site_id` | The plant the supplier ships to. |
| `delay_risk.minimum_days` | The fewest days late a shipment arrives. |
| `delay_risk.maximum_days` | The most days late a shipment arrives. |
| `delay_risk.common_risk_group` | Suppliers in the same group are late together. Volt Motors and Pacific Impeller share the West Coast port. |

The `assembly-line` row is our own line. It is listed so it can be approved to build the
pumps (see `qualifications` in the capsule). It has no delay risk.

### Stock on hand

In [3]:
!cat data/on-hand.csv

id,item_id,site_id,quantity,uom,available_at
OH-PMP-100,PMP-100,plant,6,each,2026-11-18T07:00:00-06:00
OH-MTR-05,MTR-05,plant,30,each,2026-11-18T07:00:00-06:00
OH-MTR-20,MTR-20,plant,8,each,2026-11-18T07:00:00-06:00
OH-HSG-100,HSG-100,plant,55,each,2026-11-18T07:00:00-06:00
OH-HSG-300,HSG-300,plant,40,each,2026-11-18T07:00:00-06:00
OH-IMP-100,IMP-100,plant,60,each,2026-11-18T07:00:00-06:00
OH-IMP-300,IMP-300,plant,12,each,2026-11-18T07:00:00-06:00
OH-SEAL-KIT,SEAL-KIT,plant,90,each,2026-11-18T07:00:00-06:00


| Column | Meaning |
|---|---|
| `id` | The stock record. |
| `item_id` | The part number. |
| `site_id` | Where it is. |
| `quantity`, `uom` | How many. |
| `available_at` | When it can be used: now. |

Note the 6 finished PMP-100 pumps on the shelf. The twin ships those first.

### Open purchase orders

In [4]:
!cat data/open-purchase-orders.csv

id,item_id,site_id,quantity,uom,available_at,supplier_id
PO-7781,MTR-05,plant,40,each,2026-11-27T08:00:00-06:00,volt-motors
PO-7790,MTR-20,plant,30,each,2026-12-02T08:00:00-06:00,volt-motors
PO-7795,IMP-300,plant,30,each,2026-11-30T08:00:00-06:00,pacific-impeller
PO-7802,HSG-300,plant,20,each,2026-11-24T08:00:00-06:00,midwest-foundry
PO-7810,SEAL-KIT,plant,100,each,2026-11-23T08:00:00-06:00,lakeside-seals


| Column | Meaning |
|---|---|
| `id` | The purchase order number. |
| `item_id`, `quantity`, `uom` | What is on order. |
| `available_at` | The dock date the supplier promised. The twin adds the supplier's delay. |
| `supplier_id` | Who ships it. Its delay risk comes from `data/suppliers.csv`. |

### Customer orders

In [5]:
!cat data/customer-orders.csv

id,customer,item_id,site_id,quantity,uom,due_at,priority,unit_price
SO-4410,Prairie Well Service,PMP-100,plant,20,each,2026-11-25T17:00:00-06:00,2,385
SO-4412,Big Sky Irrigation,PMP-300,plant,8,each,2026-11-27T17:00:00-06:00,3,1240
SO-4415,Hometown Hardware,PMP-100,plant,15,each,2026-12-04T17:00:00-06:00,1,385
SO-4418,Valley Farm Supply,PMP-300,plant,12,each,2026-12-08T17:00:00-06:00,3,1240
SO-4421,Prairie Well Service,PMP-100,plant,25,each,2026-12-11T17:00:00-06:00,2,385
SO-4423,Big Sky Irrigation,PMP-300,plant,14,each,2026-12-11T17:00:00-06:00,3,1240
SO-4426,Lakeshore Rentals,PMP-100,plant,10,each,2026-12-18T17:00:00-06:00,1,385


| Column | Meaning |
|---|---|
| `id` | The sales order number. |
| `customer` | Who ordered it. twinflow does not need it. |
| `item_id`, `quantity`, `uom` | What they ordered. |
| `due_at` | The promised ship date and time. |
| `priority` | Higher numbers get parts first. Farm customers (3) come before the home store (1). |
| `unit_price` | The price of one pump. twinflow does not need it. It helps you see the money at risk. |

### The twin capsule

The capsule points at each file, for example `orders: data/customer-orders.csv`. It also
holds the bill of materials (the parts in each pump) and the days to assemble an order.

In [6]:
!cat harbor-pumps.twin.yaml

# Harbor Pump Works: the twin capsule for the pump plant's supply network.
# It ties the ERP exports in data/ together: items, suppliers, stock on hand, open
# purchase orders and customer orders. Run it with:
#   twinflow scenario run harbor-pumps.twin.yaml --summary
schema_version: '0.1'
required_capabilities: []
model:
  domain: supply_chain
  sites:
    - id: plant
  suppliers: data/suppliers.csv          # who ships to us, and how late they can be
  items: data/items.csv                  # finished pumps and the parts we buy
  bom_revisions:                         # the bill of materials: parts in one pump
    - id: PMP-100-rev-C
      assembly_item_id: PMP-100
      revision: C
      effective_from: '2026-01-01T00:00:00-06:00'
      lines:
        - {item_id: MTR-05, quantity: 1, uom: each}
        - {item_id: HSG-100, quantity: 1, uom: each}
        - {item_id: IMP-100, quantity: 1, uom: each}
        - {item_id: SEAL-KIT, quantity: 1, uom: each}
    - id: PMP-300-rev-B
      a

## (c) Check and run

First, check the files. No message means everything is good.

In [7]:
!twinflow scenario validate harbor-pumps.twin.yaml

### Today's plan

This runs the network 50 times. Each run draws a new delay for every supplier. When the
port is slow in a run, both overseas suppliers are slow in that run.

- **Ready (typical)** is the middle date over the 50 runs: half the runs are ready by then.
- **Ready (bad case)** is the date by which 9 runs in 10 are ready.
- **On time** is the share of runs in which the order is ready by its due date.

In [8]:
!twinflow scenario run harbor-pumps.twin.yaml --out runs/today --summary

Ran 50 runs (seed 7).
Order    Item     Qty  Due         Ready (typical)  Ready (bad case)  On time
SO-4410  PMP-100  20   Wed Nov 25  Fri Nov 20       Fri Nov 20        100% (50 of 50)
SO-4412  PMP-300  8    Fri Nov 27  Sat Nov 21       Sat Nov 21        100% (50 of 50)
SO-4415  PMP-100  15   Fri Dec 04  Sat Dec 05       Tue Dec 08        40% (20 of 50)
SO-4418  PMP-300  12   Tue Dec 08  Fri Dec 11       Mon Dec 14        14% (7 of 50)
SO-4421  PMP-100  25   Fri Dec 11  Sat Dec 05       Tue Dec 08        100% (50 of 50)
SO-4423  PMP-300  14   Fri Dec 11  Fri Dec 11       Mon Dec 14        48% (24 of 50)
SO-4426  PMP-100  10   Fri Dec 18  never            never             0% (0 of 50)
Short of parts:
  SO-4426 needs 9 more HSG-100 (in 50 of 50 runs)
  SO-4426 needs 4 more IMP-100 (in 50 of 50 runs)
Full result: runs/today/result.json


What this says:

- The two November orders are safe. They use stock on hand.
- **SO-4426 (Lakeshore Rentals) can never ship.** There are not enough small housings and
  impellers, and none are on order. The twin names the parts and how many: 9 HSG-100 and
  4 IMP-100.
- **SO-4418 (Valley Farm Supply) is on time in only 14% of runs**, and SO-4423 in about
  half. Both wait on the 2 hp motors (PO-7790) and the large impellers (PO-7795), which
  come through the port.
- SO-4415 (Hometown Hardware) waits on the small motors (PO-7781). It is on time in 40% of runs.

### Plan B

Plan B changes only the purchase orders: both motor orders fly (earlier dock dates,
`volt-motors-air` as the shipper, 0 to 2 days of delay), and two new orders cover the
small housings and impellers. Everything else is the same.

In [9]:
!twinflow scenario validate harbor-pumps-plan-b.twin.yaml

In [10]:
!twinflow scenario run harbor-pumps-plan-b.twin.yaml --out runs/plan-b --summary

Ran 50 runs (seed 7).
Order    Item     Qty  Due         Ready (typical)  Ready (bad case)  On time
SO-4410  PMP-100  20   Wed Nov 25  Fri Nov 20       Fri Nov 20        100% (50 of 50)
SO-4412  PMP-300  8    Fri Nov 27  Sat Nov 21       Sat Nov 21        100% (50 of 50)
SO-4415  PMP-100  15   Fri Dec 04  Fri Nov 27       Sat Nov 28        100% (50 of 50)
SO-4418  PMP-300  12   Tue Dec 08  Mon Dec 07       Thu Dec 10        66% (33 of 50)
SO-4421  PMP-100  25   Fri Dec 11  Fri Nov 27       Sat Nov 28        100% (50 of 50)
SO-4423  PMP-300  14   Fri Dec 11  Mon Dec 07       Thu Dec 10        100% (50 of 50)
SO-4426  PMP-100  10   Fri Dec 18  Tue Dec 15       Fri Dec 18        94% (47 of 50)
Full result: runs/plan-b/result.json


## (d) Your rules

For a supply network, the written piece is the buyer's plan. It is two small files: the
new purchase-order file and a copy of the capsule that points at it. The boxes below save
them, so you can change a date and run the plan again.

### `data/open-purchase-orders-plan-b.csv`: the buyer's plan

Compared with `data/open-purchase-orders.csv`:

- PO-7781 and PO-7790 (motors) now ship by air: earlier dates and `volt-motors-air`.
- PO-7821 and PO-7822 are new: 20 small housings and 20 small impellers.

In [11]:
%%writefile data/open-purchase-orders-plan-b.csv
id,item_id,site_id,quantity,uom,available_at,supplier_id
PO-7781,MTR-05,plant,40,each,2026-11-24T08:00:00-06:00,volt-motors-air
PO-7790,MTR-20,plant,30,each,2026-11-27T08:00:00-06:00,volt-motors-air
PO-7795,IMP-300,plant,30,each,2026-11-30T08:00:00-06:00,pacific-impeller
PO-7802,HSG-300,plant,20,each,2026-11-24T08:00:00-06:00,midwest-foundry
PO-7810,SEAL-KIT,plant,100,each,2026-11-23T08:00:00-06:00,lakeside-seals
PO-7821,HSG-100,plant,20,each,2026-12-07T08:00:00-06:00,midwest-foundry
PO-7822,IMP-100,plant,20,each,2026-12-09T08:00:00-06:00,pacific-impeller

Overwriting data/open-purchase-orders-plan-b.csv


### `harbor-pumps-plan-b.twin.yaml`: the plan B capsule

It is a copy of `harbor-pumps.twin.yaml`. Only the `receipts` line and the `id` differ.

In [12]:
%%writefile harbor-pumps-plan-b.twin.yaml
# Harbor Pump Works, plan B: the same plant, with the buyer's changes to the open
# purchase orders (air freight for both motor orders, and new orders for small housings
# and impellers). Only the receipts line differs from harbor-pumps.twin.yaml.
schema_version: '0.1'
required_capabilities: []
model:
  domain: supply_chain
  sites:
    - id: plant
  suppliers: data/suppliers.csv          # who ships to us, and how late they can be
  items: data/items.csv                  # finished pumps and the parts we buy
  bom_revisions:                         # the bill of materials: parts in one pump
    - id: PMP-100-rev-C
      assembly_item_id: PMP-100
      revision: C
      effective_from: '2026-01-01T00:00:00-06:00'
      lines:
        - {item_id: MTR-05, quantity: 1, uom: each}
        - {item_id: HSG-100, quantity: 1, uom: each}
        - {item_id: IMP-100, quantity: 1, uom: each}
        - {item_id: SEAL-KIT, quantity: 1, uom: each}
    - id: PMP-300-rev-B
      assembly_item_id: PMP-300
      revision: B
      effective_from: '2026-01-01T00:00:00-06:00'
      lines:
        - {item_id: MTR-20, quantity: 1, uom: each}
        - {item_id: HSG-300, quantity: 1, uom: each}
        - {item_id: IMP-300, quantity: 1, uom: each}
        - {item_id: SEAL-KIT, quantity: 2, uom: each}
  processes:                             # days to assemble and test one order of pumps
    - {id: assemble-PMP-100, item_id: PMP-100, lead_time_days: 2}
    - {id: assemble-PMP-300, item_id: PMP-300, lead_time_days: 3}
  qualifications:                        # the assembly line is approved to build both pumps
    - {supplier_id: assembly-line, process_id: assemble-PMP-100, valid_from: '2025-01-01T00:00:00-06:00'}
    - {supplier_id: assembly-line, process_id: assemble-PMP-300, valid_from: '2025-01-01T00:00:00-06:00'}
  gates: []
snapshot:
  id: harbor-2026-11-18
  as_of: '2026-11-18T07:00:00-06:00'
  model_revision: '1'
  inventory: data/on-hand.csv            # stock on the shelf this morning
  receipts: data/open-purchase-orders-plan-b.csv  # the buyer's plan B purchase orders
  allocations: []
  documents: []
  orders: data/customer-orders.csv       # what customers ordered, and when it is due
experiment:
  id: harbor-plan-b
  replications: 50                       # 50 runs, each with new supplier delays
  seed: 7
assumptions:
  - Dates cover parts on hand, parts on order and assembly time. Line capacity is not modeled.
provenance:
  source: Harbor Pump Works ERP export, 2026-11-18

Overwriting harbor-pumps-plan-b.twin.yaml


## (e) Results and the decision

| Order | Customer | Value | Today: on time | Plan B: on time |
|---|---|---|---|---|
| SO-4415 | Hometown Hardware | $5,775 | 40% | 100% |
| SO-4418 | Valley Farm Supply | $14,880 | 14% | 66% |
| SO-4423 | Big Sky Irrigation | $17,360 | 48% | 100% |
| SO-4426 | Lakeshore Rentals | $3,850 | never | 94% |

**The recommendation:** do plan B today.

- Air freight costs about $3,800. It protects about $42,000 of orders that are at risk
  today, and the farm customers who buy every spring.
- Place the housing and impeller orders now. Without them SO-4426 cannot ship at all.
- **SO-4418 is still at risk** (66%). Its large impellers (PO-7795) come through the port.
  Ask Pacific Impeller to air-ship 12 of the 30, or call Valley Farm Supply now with a
  likely date of December 7 and a bad case of December 10.

The full result of each run, with every shortage and its parts path, is in
`runs/today/result.json` and `runs/plan-b/result.json`. Both are made when you run the
notebook.

## (f) Try it yourself

1. **Fly the impellers too.** In `data/open-purchase-orders-plan-b.csv`, change PO-7795's
   supplier to a new air row you add in `data/suppliers.csv` (copy the `volt-motors-air`
   row and name it `pacific-impeller-air`), and move its date to 2026-11-25. Run the plan B
   boxes again. Is SO-4418 safe now?
2. **A worse port.** In `data/suppliers.csv`, change Volt Motors' `maximum_days` from 10
   to 15. Run today's plan again. Which orders move?
3. **A new order.** Add a row to `data/customer-orders.csv`:
   `SO-4430,Valley Farm Supply,PMP-300,plant,6,each,2026-12-15T17:00:00-06:00,3,1240`.
   Run both plans. Which order gives up its parts?